In [1]:
import torch
import triton
import triton.language as tl

@triton.jit
def multiply_kernel(A_ptr, N, BLOCK_SIZE: tl.constexpr):
    # 获取当前程序实例（Program）的 ID，等价于线程编号
    # 类似于 POSIX 中的：pthread_t tid = pthread_self();
    pid = tl.program_id(0)

    # 计算当前程序负责处理的数据索引范围
    # tl.arange(0, BLOCK_SIZE) 返回一个向量 [0, 1, ..., BLOCK_SIZE-1]
    # 每个 program 处理 BLOCK_SIZE 个元素，偏移量取决于 pid
    # 例如：pid=1 → offsets = [32, 33, ..., 63]
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

    # 创建掩码（mask），用于屏蔽越界访问
    # 只处理 offsets 中小于 N 的合法索引（N 是数组总长度）
    mask = offsets < N

    # 从全局内存中加载对应索引位置的数据
    # A_ptr 是数组 A 的指针，offsets 是该 program 负责的偏移
    # 加上 mask 是为了避免访问越界位置
    # a 变量所在的位置是program自己的私有sram
    a = tl.load(A_ptr + offsets, mask=mask)

    # 对数据进行逐元素乘以 2 的操作
    a *= 2

    # 将结果写回原始位置，仍使用 mask 避免越界写入
    tl.store(A_ptr + offsets, a, mask=mask)

In [18]:
# 🔢 准备输入数据：逆序 [100, 99, ..., 1]
x = torch.arange(100, 0, -1, dtype=torch.float32, device='cuda')
N = x.numel()
MY_BLOCK_SIZE = 32

# 🧮 计算启动多少个 program（每个处理 MY_BLOCK_SIZE 个元素）
# Triton 启动网格必须是元组格式 (1D, )，哪怕是一维也不能省略逗号
grid = (triton.cdiv(N, MY_BLOCK_SIZE),)

# 解释：grid = ((N + MY_BLOCK_SIZE - 1) // MY_BLOCK_SIZE,)
# 对于 N = 100，BLOCK_SIZE = 32，结果是 4，最后一个 program 只处理 4 个元素

# 🧠 可选写法：使用 lambda 动态计算（推荐用于自动调优）
# grid = lambda meta: (triton.cdiv(N, meta['BLOCK_SIZE']),)

In [ ]:
# 🚀 启动 Triton kernel
multiply_kernel[grid](
    x,              # A_ptr：PyTorch Tensor 会自动转换为设备指针传入 GPU
    N,              # N：待处理元素数量
    MY_BLOCK_SIZE
)

In [20]:
print(x)

tensor([200., 198., 196., 194., 192., 190., 188., 186., 184., 182., 180., 178.,
        176., 174., 172., 170., 168., 166., 164., 162., 160., 158., 156., 154.,
        152., 150., 148., 146., 144., 142., 140., 138., 136., 134., 132., 130.,
        128., 126., 124., 122., 120., 118., 116., 114., 112., 110., 108., 106.,
        104., 102., 100.,  98.,  96.,  94.,  92.,  90.,  88.,  86.,  84.,  82.,
         80.,  78.,  76.,  74.,  72.,  70.,  68.,  66.,  64.,  62.,  60.,  58.,
         56.,  54.,  52.,  50.,  48.,  46.,  44.,  42.,  40.,  38.,  36.,  34.,
         32.,  30.,  28.,  26.,  24.,  22.,  20.,  18.,  16.,  14.,  12.,  10.,
          8.,   6.,   4.,   2.], device='cuda:0')
